In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [23]:
!pip install -q transformers datasets sentencepiece sacrebleu

In [24]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
) 

In [7]:
dataset = load_dataset("thainq107/iwslt2015-en-vi")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 133317
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
})


In [8]:
train_dataset = dataset["train"].select(range(25000))
valid_dataset = dataset["validation"].select(range(1000))

print(len(train_dataset))
print(len(valid_dataset)) 

25000
1000


In [43]:
model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

In [44]:
prefix = "translate English to Vietnamese: "

def preprocess(example):

    inputs = [prefix + x for x in example["en"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=example["vi"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [45]:
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess,
    batched=True,
    remove_columns=valid_dataset.column_names
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [42]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./model",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    num_train_epochs=5,

    predict_with_generate=True,

    logging_steps=200,

    save_total_limit=2,

    fp16=True
) 

In [37]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    data_collator=data_collator
)

In [38]:
trainer.train()

Step,Training Loss
200,1.737716
400,1.752814
600,1.737961
800,1.732123
1000,1.718614
1200,1.712163
1400,1.706804
1600,1.703466
1800,1.690695
2000,1.693131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15625, training_loss=1.6089279267578125, metrics={'train_runtime': 1387.6151, 'train_samples_per_second': 90.083, 'train_steps_per_second': 11.26, 'total_flos': 2092921388531712.0, 'train_loss': 1.6089279267578125, 'epoch': 5.0})

In [53]:
text = "Sometimes I feel like my only friend."

input_text = "translate English to Vietnamese: " + text

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_length=64,
    num_beams=5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))  

Tiếng Anh sang tiếng Việt: tôi cảm thấy như người bạn duy nhất của tôi.


In [54]:
sentences = [
    "I love machine learning.",
    "This is a beautiful day.",
    "We will meet tomorrow.",
    "Artificial intelligence is changing the world."
]

for s in sentences:
    input_text = "translate English to Vietnamese: " + s
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_length=64, num_beams=5)

    print("EN:", s)
    print("VI:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print() 

EN: I love machine learning.
VI: Dịch ra tiếng Anh: tôi yêu máy học.

EN: This is a beautiful day.
VI: Tiếng Anh sang: đây là một ngày đẹp trời.

EN: We will meet tomorrow.
VI: Từ tiếng Anh sang tiếng Việt: chúng ta sẽ gặp nhau vào ngày mai.

EN: Artificial intelligence is changing the world.
VI: Từ tiếng Anh sang tiếng Việt: tình báo đang thay đổi thế giới.



In [56]:
model.save_pretrained("/kaggle/working/envi-model")
tokenizer.save_pretrained("/kaggle/working/envi-model") 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/envi-model/tokenizer_config.json',
 '/kaggle/working/envi-model/vocab.json',
 '/kaggle/working/envi-model/source.spm',
 '/kaggle/working/envi-model/target.spm',
 '/kaggle/working/envi-model/added_tokens.json')

In [57]:
!zip -r envi-model.zip /kaggle/working/envi-model

  adding: kaggle/working/envi-model/ (stored 0%)
  adding: kaggle/working/envi-model/generation_config.json (deflated 43%)
  adding: kaggle/working/envi-model/target.spm (deflated 50%)
  adding: kaggle/working/envi-model/config.json (deflated 63%)
  adding: kaggle/working/envi-model/vocab.json (deflated 70%)
  adding: kaggle/working/envi-model/source.spm (deflated 51%)
  adding: kaggle/working/envi-model/model.safetensors (deflated 7%)
  adding: kaggle/working/envi-model/tokenizer_config.json (deflated 68%)
